In [3]:
#Tarea 3: Manejo de Valores Nulos y Duplicados

#Objetivo: Implementar estrategias avanzadas para datos faltantes y eliminar duplicados.

### Ejercicios:
#- Analizar patrones de nulos (¿hay correlación entre columnas?).
#- Para `customer_id` nulos, intentar inferir por transacciones de la misma tienda/fecha.
#- Para `amount` nulos, probar diferentes estrategias: media, mediana, valor anterior.
#- Decidir cuál estrategia es mejor y justificar por qué.
#- Eliminar duplicados exactos y por `transaction_id`.
#- Enriquecer datos agregando columnas:
#    - `mes` (extraído de fecha)
#    - `trimestre`
#    - `dia_semana`
#    - `rango_monto` (bajo, medio, alto)
#- Guardar resultado como `transacciones_enriched.csv`.

In [16]:
import os
import sys
from pathlib import Path
from datetime import datetime
# Verificar el intérprete de Python activo
print(f"Intérprete Python: {sys.executable}")
print(f"Versión Python: {sys.version}")

# Mostrar directorio de trabajo actual
current_dir = os.getcwd()
print(f"\nDirectorio actual: {current_dir}")

# Ruta del proyecto dentro del contenedor Docker
# El docker-compose monta ./ en /home/jovyan/work
project_root = Path("/home/jovyan/work")

# Cambiar al directorio del proyecto si es necesario
if os.getcwd() != str(project_root):
    os.chdir(project_root)
    print(f"Directorio cambiado a: {os.getcwd()}")
else:
    print(f"Ya estamos en el directorio del proyecto: {project_root}")

# Agregar el proyecto a sys.path para imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Ruta del proyecto agregada a sys.path")

print(f"\nRaíz del proyecto: {project_root}")


Intérprete Python: /opt/conda/bin/python
Versión Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]

Directorio actual: /home/jovyan/work
Ya estamos en el directorio del proyecto: /home/jovyan/work

Raíz del proyecto: /home/jovyan/work


In [17]:
# Validar que las carpetas existen
# Dentro de Docker: /home/jovyan/work/homeworks/...
homeworks_dir = project_root / "homeworks"
data_dir = project_root / "data/processed"
output_dir = homeworks_dir / "output"
tarea_3_dir = homeworks_dir / "tarea_3"

print("Validando estructura de carpetas (rutas dentro del contenedor Docker):")
print(f"  project_root : {project_root}")
print(f"  homeworks/   : {homeworks_dir}  → existe: {homeworks_dir.exists()}")
print(f"  data/        : {data_dir}  → existe: {data_dir.exists()}")
print(f"  output/      : {output_dir}  → existe: {output_dir.exists()}")
print(f"  tarea_3/     : {tarea_3_dir}  → existe: {tarea_3_dir.exists()}")

# Crear output si no existe
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n✓ Carpeta output creada: {output_dir}")
else:
    print(f"\n✓ Carpeta output ya existe")


Validando estructura de carpetas (rutas dentro del contenedor Docker):
  project_root : /home/jovyan/work
  homeworks/   : /home/jovyan/work/homeworks  → existe: True
  data/        : /home/jovyan/work/data/processed  → existe: True
  output/      : /home/jovyan/work/homeworks/output  → existe: True
  tarea_3/     : /home/jovyan/work/homeworks/tarea_3  → existe: True

✓ Carpeta output ya existe


In [18]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import re

print("✓ Librerías importadas correctamente:")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")

✓ Librerías importadas correctamente:
  - pandas 2.1.1
  - numpy 1.24.4


In [19]:
# Cargar datos limpios desde la tarea 2
csv_path = data_dir / "transacciones_clean.csv"
print(f"Cargando datos desde: {csv_path}")

if csv_path.exists():
    df_raw = pd.read_csv(csv_path)
    print(f"✓ Datos cargados exitosamente")
    print(f"  - Filas: {len(df_raw)}")
    print(f"  - Columnas: {len(df_raw.columns)}")
    print(f"\nPrimeras filas del dataset raw:")
    print(df_raw.head())
else:
    print(f"✗ Archivo no encontrado: {csv_path}")
    print("Verifica que la Tarea 1 fue ejecutada correctamente")

# Guardar copia del original para comparación
df_backup = df_raw.copy()


Cargando datos desde: /home/jovyan/work/data/processed/transacciones_clean.csv
✓ Datos cargados exitosamente
  - Filas: 105
  - Columnas: 6

Primeras filas del dataset raw:
  transaction_id        date customer_id  amount     status  \
0      TXN-00001  2023-03-02        1038  194.07        NaN   
1      TXN-00002  2023-04-06        1002  452.64  CANCELADA   
2      TXN-00003  2023-08-24        1017  344.76    FALLIDA   
3      TXN-00004  2023-01-20        1041   85.16        NaN   
4      TXN-00005  02-07-2023        1000   72.53    FALLIDA   

                    store  
0  TIENDA_SIN_ESPECIFICAR  
1            Tienda_Norte  
2              Tienda_Sur  
3              Tienda_Sur  
4            Tienda_Norte  
